In [81]:
import torch
import torch.nn.functional as F
from ultralytics import YOLO
from torchvision.ops import box_iou
from tqdm import tqdm
import cv2
import numpy as np
import os

In [83]:
teacher = YOLO('src/training_yolo/yolo11n_half.pt')
student = YOLO('src/training_yolo/yolo11n_full.pt')

teacher_model = teacher.model.cuda().eval()
student_model = student.model.cuda().train()



In [84]:
teacher_model.eval()

for p in teacher_model.parameters(): 
    p.requirs_grad = False

In [129]:
student_model.train()


for p in student_model.parameters():
    p.requires_grad = True

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
teacher_model.to(device)
student_model.to(device)

In [131]:
bbox_loss_fn = torch.nn.SmoothL1Loss()

In [132]:
optimizer = torch.optim.Adam(student_model.model.parameters(), lr=1e-6)

In [133]:
dataset_path = "datasets/images/val2017"

In [134]:
image_files = [os.path.join(dataset_path, f) for f in os.listdir(dataset_path) if f.endswith(('.jpg', '.png'))]

In [135]:
image_files[0]

'datasets/images/val2017/000000536343.jpg'

In [136]:
def load_image_tensor(path):
    img = cv2.imread(path)
    img = cv2.resize(img, (640, 640)) 
    img = img[:, :, ::-1]  # BGR to RGB
    img = img / 255.0 # scale to [0,1]
    img = torch.from_numpy(img).permute(2, 0, 1).float().unsqueeze(0) # HWC to CHW
    return img.cuda()

In [137]:
def match_boxes(student_boxes, teacher_boxes, iou_threshold=0.5):
    if student_boxes.size(0) == 0 or teacher_boxes.size(0) == 0:
        return torch.empty((0, 4), device=student_boxes.device), torch.empty((0, 4), device=teacher_boxes.device)
    
    ious = box_iou(student_boxes, teacher_boxes)
    max_iou, indices = ious.max(dim=1)
    mask = max_iou > iou_threshold
    
    return student_boxes[mask], teacher_boxes[indices[mask]]

In [138]:
for epoch in range(1):
    print(f"Epoch {epoch + 1}")
    total_loss = 0.0
    pbar = tqdm(image_files)

    for image_path in pbar:
        image = load_image_tensor(image_path)

        with torch.no_grad():
            teacher_raw_out = teacher_model(image)
            teacher_raw_preds = teacher_raw_out[0]  # shape: (1, 84, 8400)
            teacher_xywh = teacher_raw_preds[..., :4]  # (x_center, y_center, w, h)

            teacher_xyxy = torch.zeros_like(teacher_xywh)
            teacher_xyxy[..., 0] = teacher_xywh[..., 0] - teacher_xywh[..., 2] / 2  # x1
            teacher_xyxy[..., 1] = teacher_xywh[..., 1] - teacher_xywh[..., 3] / 2  # y1
            teacher_xyxy[..., 2] = teacher_xywh[..., 0] + teacher_xywh[..., 2] / 2  # x2
            teacher_xyxy[..., 3] = teacher_xywh[..., 1] + teacher_xywh[..., 3] / 2  # y2

        student_out = student_model(image)
        raw_preds = student_out[0]  # (1, 84, 8400)

        print(raw_preds.shape, teacher_raw_preds.shape)
        student_xywh = raw_preds[..., :4]  # get box predictions

        student_xyxy = torch.zeros_like(student_xywh)
        student_xyxy[..., 0] = student_xywh[..., 0] - student_xywh[..., 2] / 2  # x1
        student_xyxy[..., 1] = student_xywh[..., 1] - student_xywh[..., 3] / 2  # y1
        student_xyxy[..., 2] = student_xywh[..., 0] + student_xywh[..., 2] / 2  # x2
        student_xyxy[..., 3] = student_xywh[..., 1] + student_xywh[..., 3] / 2  # y2


        # print(student_out.requires_grad)
        student_matched, teacher_matched = match_boxes(student_boxes, teacher_boxes, iou_threshold=0.5)


        print(student_matched.shape, teacher_matched.shape, student_matched.requires_grad)
        if len(student_matched) == 0:
            continue  # skip if no matched pairs

        # ----- Compute loss -----
        loss = bbox_loss_fn(student_matched, teacher_matched)

        # ----- Backward pass -----
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # ----- Logging -----
        total_loss += loss.item()
        pbar.set_description(f"Loss: {loss.item():.4f}")

    print(f"Epoch {epoch + 1} - Total Loss: {total_loss:.4f}")


Epoch 1


  0%|                                                                                                                                                                                                                | 0/5000 [00:00<?, ?it/s]

torch.Size([1, 144, 80, 80]) torch.Size([1, 144, 80, 80])


NameError: name 'student_boxes' is not defined

In [ ]:
student_matched.requires_grad

In [ ]:
from pathlib import Path
output_dir = Path('runs/pseudolabels')
output_dir.mkdir(exist_ok=True)
teacher.predict(source='datasets/images/train2017', save_txt=True, project=output_dir, name='pseudo')

In [ ]:
import torch.nn.functional as F
from ultralytics import YOLO
from ultralytics.models.yolo.detect.train import DetectionTrainer
from ultralytics.utils.loss import v8DetectionLoss
from ultralytics.utils import LOGGER
import logging
LOGGER.setLevel(logging.INFO)

class MyKDDetectionTrainer(DetectionTrainer):
    def __init__(self, student_model, teacher_model, overrides=None):
        super().__init__(overrides=overrides)
        self.teacher = teacher_model
        self.teacher.model
        self.model = student_model

        self.loss_fn = v8DetectionLoss(student_model)

    def get_model(self, cfg=None, weights=None):
        return self.model

    def loss(self, preds, batch):
        yolo_loss = self.loss_fn(preds, batch)
      

        with torch.no_grad():
            teacher_preds = self.teacher.model(batch['img'])[0]
        student_preds = preds[0] if isinstance(preds, (list, tuple)) else preds
        kd_loss = F.mse_loss(student_preds, teacher_preds)
        LOGGER.info(f"[KD] YOLO Loss: {yolo_loss.item():.4f}, KD Loss: {kd_loss.item():.4f}")
        return 0.7 * yolo_loss + 0.3 * kd_loss


def train_kd_yolo(
    teacher_weights,
    student_weights,
    data,
    epochs=50,
    batch=16,
    imgsz=640,
    device='cuda' if torch.cuda.is_available() else 'cpu'
):
    teacher = YOLO(teacher_weights)
    student = YOLO(student_weights)
    teacher.model.eval()
    for p in teacher.model.parameters():
        p.requires_grad = False

    overrides = {
        'model': student_weights,
        'data': data,
        'epochs': epochs,
        'batch': batch,
        'imgsz': imgsz,
        'device': device,
    }

    trainer = MyKDDetectionTrainer(
        student_model=student.model,
        teacher_model=teacher,
        overrides=overrides
    )

    trainer.train()
    return student

if __name__ == "__main__":
    trained_student = train_kd_yolo(
        teacher_weights='models/yolov8n.pt',  # Make sure this path is correct
        student_weights='models/yolo11n.pt',  # Use .yaml for architecture
        data='my_config.yaml',  # Make sure this path is correct
        epochs=5,
        batch=16,
        imgsz=640
    )
    print("✅ Knowledge distillation completed!")

In [207]:
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils.loss import v8DetectionLoss
from ultralytics import YOLO
import torch
import torch.nn as nn
import torch.nn.functional as F
from ultralytics.utils import DEFAULT_CFG

class KnowledgeDistillationLoss(v8DetectionLoss):
    """Custom loss function combining YOLO detection loss with knowledge distillation."""
    
    def __init__(self, model, teacher_model=None, alpha=0.7, temperature=4.0):
        super().__init__(model)
        self.model = model
        self.teacher_model = teacher_model
        self.alpha = alpha  # Balance between task loss and distillation loss
        self.temperature = temperature
        self.nc = model.nc 
            
        if self.teacher_model:
            self.teacher_model.eval()
            for param in self.teacher_model.parameters():
                param.requires_grad = False
    
    def __call__(self, preds, batch):
        """Compute combined loss: YOLO detection loss + knowledge distillation loss."""
        # Standard YOLO detection loss
        detection_result = super().__call__(preds, batch)
        detection_loss, detection_loss_items = detection_result
        detection_loss.sum()
        
        if self.teacher_model is None:
            return detection_loss
            
        with torch.no_grad():
            teacher_preds = self.teacher_model(batch['img'])

        # Compute distillation loss
        distillation_loss = self.compute_distillation_loss(preds, teacher_preds)
        
        # Combine losses
        print(detection_loss, distillation_loss)
        total_loss = self.alpha * detection_loss + (1 - self.alpha) * distillation_loss
        
        return total_loss, detection_loss_items
    
    def extract_spatial_format(self, pred):
        """
        Extract components from YOLO prediction tensor
        pred: [batch, channels, height, width] where channels = (4 + 1 + nc) * num_anchors
        """
        print(f"Processing tensor shape: {pred.shape}")
        batch, channels, height, width = pred.shape
    

        # For COCO with 80 classes, standard format is 85 channels per anchor

        num_anchors = 3 
        attrs_per_anchor = 48 
            # print(f"Using 3 anchors with 48 channels each")
                
        # Reshape to [batch, num_anchors, attrs_per_anchor, height, width]
        reshaped = pred.view(batch, num_anchors, attrs_per_anchor, height, width)
        
        # Extract components - assuming first 5 channels are bbox + objectness
        bbox_coords = reshaped[:, :, :4, :, :]      # First 4 channels: bbox
        objectness = reshaped[:, :, 4:5, :, :]      # 5th channel: objectness
        
        # Remaining channels as class predictions (43 channels)
        # This might be a compressed class representation
        class_preds = reshaped[:, :, 5:, :, :]      # Remaining 43 channels
        
        return bbox_coords, objectness, class_preds
        
    def extract_flattened_format(self, pred):
        """Extract components from flattened format [B, C, N] where N is total grid cells"""
        batch, channels, num_cells = pred.shape
        print(f"Processing flattened tensor: {pred.shape}")
        
        # For COCO: typically 84 channels = 3 anchors × (4 bbox + 1 obj + 15 classes)
        # or similar compressed format
        
        # Assume 3 anchors with 28 channels each (4+1+23) - compressed classes
        num_anchors = 3
        attrs_per_anchor = 28
    
        # print(f"Flattened format: {num_anchors} anchors with {attrs_per_anchor} channels each")
        
        # Reshape to [batch, num_anchors, attrs_per_anchor, num_cells]
        reshaped = pred.view(batch, num_anchors, attrs_per_anchor, num_cells)
        
        bbox_coords = reshaped[:, :, :4, :]      # [B, anchors, 4, num_cells]
        objectness = reshaped[:, :, 4:5, :]      # [B, anchors, 1, num_cells]
        class_preds = reshaped[:, :, 5:, :]      # [B, anchors, remaining, num_cells]
        
        return bbox_coords, objectness, class_preds
    
    
    def compute_distillation_loss(self, student_preds, teacher_preds):
        """Compute distillation loss between student and teacher outputs."""
        total_kd_loss = 0.0 
        processed_heads = 0

        min_heads = min(len(student_preds), len(teacher_preds))
        
        for i in range(min_heads): 
            s_pred = student_preds[i]
            t_pred = teacher_preds[i]

            if isinstance(t_pred, list):
                continue 
                

            # print(f"Head {i}: Student {s_pred.shape}, Teacher {t_pred.shape}")

            s_bbox, s_obj, s_cls = self.extract_spatial_format(s_pred) 
            t_bbox, t_obj, t_cls = self.extract_flattened_format(t_pred)
                
            # print(f"Student - bbox: {s_bbox.shape}, obj: {s_obj.shape}, cls: {s_cls.shape}")
            # print(f"Teacher - bbox: {t_bbox.shape}, obj: {t_obj.shape}, cls: {t_cls.shape}")

            # Flatten spatial dimension of studente 80 * 80 = 6400 
            s_bbox_flat = s_bbox.view(s_bbox.shape[0], s_bbox.shape[1], s_bbox.shape[2], -1)  
            s_obj_flat = s_obj.view(s_obj.shape[0], s_obj.shape[1], s_obj.shape[2], -1)     
            s_cls_flat = s_cls.view(s_cls.shape[0], s_cls.shape[1], s_cls.shape[2], -1)  

             # Handle different spatial resolutions
            student_cells = s_bbox.shape[-1]   
            teacher_cells = t_bbox.shape[-1]  


            # Interpolate student features to match teacher resolution 
            scale_factor = teacher_cells / student_cells  
            
             # Reshape back to spatial for interpolation
            print(s_bbox.shape)
            print(s_bbox.shape[3], s_bbox.shape[4])
            H, W = s_bbox.shape[3], s_bbox.shape[4]  # Student spatial dimensions
            new_H = int(H * (scale_factor ** 0.5))
            new_W = int(W * (scale_factor ** 0.5))

            # Interpolate bbox coordinates
            s_bbox_spatial = s_bbox_flat.view(s_bbox_flat.shape[0], s_bbox_flat.shape[1], s_bbox_flat.shape[2], H, W)            
            s_bbox_interp = F.interpolate(s_bbox_spatial.view(-1, s_bbox_spatial.shape[2], H, W), 
                                        size=(new_H, new_W), mode='bilinear', align_corners=False)
            s_bbox_final = s_bbox_interp.view(s_bbox_flat.shape[0], s_bbox_flat.shape[1], s_bbox_flat.shape[2], -1)

             # Interpolate objectness
            s_obj_spatial = s_obj_flat.view(s_obj_flat.shape[0], s_obj_flat.shape[1], s_obj_flat.shape[2], H, W)
            s_obj_interp = F.interpolate(s_obj_spatial.view(-1, s_obj_spatial.shape[2], H, W), 
                                       size=(new_H, new_W), mode='bilinear', align_corners=False)
            s_obj_final = s_obj_interp.view(s_obj_flat.shape[0], s_obj_flat.shape[1], s_obj_flat.shape[2], -1)
            

            # Trim or pad to exact teacher size if needed
            target_cells = teacher_cells
            current_cells = s_bbox_final.shape[-1]
            
            if current_cells > target_cells:
                s_bbox_final = s_bbox_final[..., :target_cells]
                s_obj_final = s_obj_final[..., :target_cells]
            elif current_cells < target_cells:
                pad_size = target_cells - current_cells
                s_bbox_final = F.pad(s_bbox_final, (0, pad_size))
                s_obj_final = F.pad(s_obj_final, (0, pad_size))
                
            print(f"Final shapes - Student: bbox {s_bbox_final.shape}")


            # 1. OBJECTNESS DISTILLATION (helps bbox learning)
            s_obj_prob = torch.sigmoid(s_obj_final / self.temperature)
            t_obj_prob = torch.sigmoid(t_obj / self.temperature)

            print(s_obj_prob.shape, t_obj_prob.shape)
            obj_loss = F.mse_loss(s_obj_prob, t_obj_prob)
    
            # 2. BBOX DISTILLATION (main goal)
            # Use teacher's objectness as confidence mask
            t_obj_sigmoid = torch.sigmoid(t_obj)
            confidence_mask = (t_obj_sigmoid > 0.5).float()  
    
            s_bbox_masked = s_bbox_final * confidence_mask
            t_bbox_masked = t_bbox * confidence_mask
            bbox_loss = F.mse_loss(s_bbox_masked, t_bbox_masked)
    
            # Combine only bbox + objectness losses
            head_kd_loss = bbox_loss + 0.5 * obj_loss  # Weight objectness less than bbox
            
            total_kd_loss += head_kd_loss
            processed_heads += 1

            print(f"Head {i} losses - bbox: {bbox_loss:.4f}, obj: {obj_loss:.4f}")

        final_loss = total_kd_loss / max(processed_heads, 1)
        print(f"Final bbox distillation loss: {final_loss:.4f}")
        
        return final_loss
        
        
class KnowledgeDistillationModel(DetectionModel):
    """Custom DetectionModel that uses knowledge distillation loss."""
    
    def __init__(self, cfg, teacher_model_path=None, **kwargs):
        super().__init__(cfg, **kwargs)
        self.teacher_model_path = teacher_model_path
        
    def init_criterion(self):
        """Initialize custom loss function with teacher model."""
        teacher_model = None
        if self.teacher_model_path:
            teacher_model = YOLO(self.teacher_model_path).model
            teacher_model = teacher_model.half().to('cuda')
            
        return KnowledgeDistillationLoss(self, teacher_model=teacher_model)

class KnowledgeDistillationTrainer(DetectionTrainer):
    """Custom DetectionTrainer for knowledge distillation."""
    
    def __init__(self, cfg, teacher_model_path=None, **kwargs):
        self.teacher_model_path = teacher_model_path
        super().__init__(cfg, **kwargs)
    
    def get_model(self, cfg, weights=None, verbose = False):
        """Override to return custom model with knowledge distillation."""        
        model = KnowledgeDistillationModel(
            cfg, 
            teacher_model_path=self.teacher_model_path,
            nc=self.data["nc"], 
            ch=self.data["channels"]
        )
        
        if weights:
            model.load(weights)
            
        return model
    
    def label_loss_items(self, loss_items=None, prefix="train"):
        """Override to handle custom loss components."""
        # Standard detection loss names plus distillation loss
        keys = ["box_loss", "cls_loss", "dfl_loss", "distill_loss"]
        
        if loss_items is not None:
            loss_items = [round(float(x), 5) for x in loss_items] if not isinstance(loss_items, list) else loss_items
            return dict(zip([f"{prefix}/{x}" for x in keys], loss_items))
        else:
            return keys

# Usage example
def train_with_knowledge_distillation():
    """Example of how to use the knowledge distillation trainer."""
    
    # Initialize trainer with teacher model
    trainer = KnowledgeDistillationTrainer(
        cfg=DEFAULT_CFG,
        teacher_model_path="models/yolov8n.pt",  
        overrides={
            "model": "models/yolo11n.pt", 
            "data": "my_config.yaml",
            "epochs": 1,
            "batch": 16,
            "device": "cuda"
        }
    )
    
    # Train the model
    trainer.train()
    
    # Get the trained model
    trained_model = trainer.best

# Alternative: Using YOLO interface with custom trainer
def train_with_yolo_interface():
    """Alternative approach using YOLO interface."""
    
    # Initialize student model
    student_model = YOLO("models/yolo11n.pt")
    
    # Train with custom trainer
    results = student_model.train(
        data="my_config.yaml",
        epochs=1,
        batch=16,
        trainer=KnowledgeDistillationTrainer,
        teacher_model_path="models/yolov8n.pt"
    )
    
    return results

if __name__ == "__main__":
    # Run knowledge distillation training
    train_with_knowledge_distillation()

Ultralytics 8.3.154 🚀 Python-3.10.12 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce RTX 2080 Ti, 11012MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=my_config.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=models/yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train138, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, 

COMET INFO: An experiment with the same configuration options is already running and will be reused.



                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultralytics.nn.modules.block.C3k2            [128, 128, 1, True]           
  7                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128

train: Scanning /home/achen/yolo-negative-flip/datasets/labels/train2017.cache... 1 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1/1 [00:00<?, ?it/s]


val: Fast image access ✅ (ping: 0.0±0.1 ms, read: 641.8±215.4 MB/s, size: 133.3 KB)


...

        1/1      1.63G      1.528      4.422      1.337         10        640: 100%|██████████| 1/1 [00:00<00:00,  2.86it/s]


Processing tensor shape: torch.Size([1, 144, 80, 80])
Processing flattened tensor: torch.Size([1, 84, 8400])
torch.Size([1, 3, 4, 80, 80])
80 80
Final shapes - Student: bbox torch.Size([1, 3, 4, 8400])
torch.Size([1, 3, 1, 8400]) torch.Size([1, 3, 1, 8400])
Head 0 losses - bbox: 5.9119, obj: 0.0972
Final bbox distillation loss: 5.9605
tensor([1.5277, 4.4221, 1.3372], device='cuda:0', grad_fn=<MulBackward0>) tensor(5.9605, device='cuda:0', grad_fn=<DivBackward0>)


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95):   0%|          | 0/157 [00:00<?, ?it/s]

Processing tensor shape: torch.Size([32, 84, 4410])


ValueError: not enough values to unpack (expected 4, got 3)